# Voice activity detection: an online model and an offline one

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/espnet/notebook/blob/master/Demos/diar_vad_demo.ipynb) [![diar_vad_demo](https://github.com/espnet/notebook/actions/workflows/diar_vad_demo.yml/badge.svg)](https://github.com/espnet/notebook/actions/workflows/diar_vad_demo.yml)

Find where someone is speaking, frame by frame (8 ms). Two small temporal
convolutional networks, trained with ESPnet as one-speaker diarization
(`egs2/frontia_vad_v1/diar1`), do it two ways: a causal **online** model that reads
no future audio and so runs on a live stream, and an **offline** model that
reads up to a second ahead and places the boundaries better when the whole
file is there. Here they run offline over a recording, online as audio
arrives, and offline in batches over many files, as for cleaning a corpus.

CPU, and each model is under 1 MB.

## Install

In [ ]:
%pip install -q "espnet==202612" espnet_model_zoo librosa matplotlib

## The models

`VAD_MODEL` (offline) and `VAD_ONLINE_MODEL` (online) take a Hugging Face tag
or the `.zip` that stage 8 of `diar.sh` packs, so checkpoints you trained
yourself load the same way. Each model comes with its own decision rule,
tuned on AMI MH dev.

In [ ]:
import os

from espnet2.bin.diar_inference import VoiceActivityDetector, format_segments

offline = VoiceActivityDetector.for_mode("offline", os.environ.get("VAD_MODEL"))
online = VoiceActivityDetector.for_mode("online", os.environ.get("VAD_ONLINE_MODEL"))
fs = offline.fs
for name, vad in (("online", online), ("offline", offline)):
    print(f"{name:8s} reads {vad.offline_look_ahead * vad.frame_shift:.2f} s ahead, "
          f"threshold {vad.threshold}, median filter {vad.median} frames")

## A recording with pauses

Two sentences from two speakers, with silence before, between and after.

In [ ]:
import numpy as np
from IPython.display import Audio, display

from espnet2.bin.diar_inference import read_audio

!wget -q -O a.wav https://github.com/espnet/espnet/raw/master/test_utils/ctc_align_test.wav
!wget -q -O b.wav https://github.com/espnet/espnet/raw/master/test_utils/st_test.wav

a, b = read_audio("a.wav", fs), read_audio("b.wav", fs)

def silence(seconds):
    return np.zeros(int(seconds * fs), dtype=np.float32)

recording = np.concatenate([silence(0.5), a, silence(2.0), b, silence(1.0)])
print(f"{len(recording) / fs:.1f} s")
display(Audio(recording, rate=fs))

## Offline

Each frame's speech probability from the offline model, then the decision
rule the recipe scores with: a frame is speech when its probability is above
the threshold, and a median filter removes short blips.

In [ ]:
import matplotlib.pyplot as plt

probs = offline.probabilities(recording, fs, look_ahead=offline.offline_look_ahead)
segments = offline.segments(probs)
print(format_segments(segments))

seconds = np.arange(len(recording)) / fs
fig, (top, bottom) = plt.subplots(2, 1, figsize=(10, 4), sharex=True)
top.plot(seconds, recording, linewidth=0.4, color="0.3")
bottom.plot(np.arange(len(probs)) * offline.frame_shift, probs)
bottom.axhline(offline.threshold, color="0.5", linestyle="--")
for s in segments:
    top.axvspan(s.start, s.end, alpha=0.15)
    bottom.axvspan(s.start, s.end, alpha=0.15)
bottom.set_xlabel("seconds")
bottom.set_ylabel("speech")
plt.show()

## Online

Feed the audio 100 ms at a time to the online model, as a microphone would.
Each frame is decided as soon as its samples are in: the stream keeps the
causal encoder's past and runs only the new frames, so it gets the
probabilities the whole recording gives the same model.

In [ ]:
stream = online.stream()
chunk = fs // 10
for start in range(0, len(recording), chunk):
    new = stream.accept(recording[start : start + chunk])

streamed = stream.all_probabilities()
whole = online.probabilities(recording, fs)
print("frames:", len(streamed), "largest difference:", float(np.abs(streamed - whole).max()))
print(format_segments(stream.segments()))

Compare these segments with the offline ones above: the online model has only
the past to go on, which is the price of deciding each frame as it arrives.

## Many files

`detect_files` reads files and runs them through the model in batches of
similar length (`DiarizeSpeech.diarize_batch`). Each file gets the segments
it gets alone.

In [ ]:
import soundfile

paths = []
for i, piece in enumerate([a, b, recording]):
    path = f"piece{i}.wav"
    soundfile.write(path, piece, fs)
    paths.append(path)

for path, found in zip(paths, offline.detect_files(paths, batch_size=3)):
    speech = sum(s.duration for s in found)
    print(f"{path}: {len(found)} segments, {speech:.2f} s of speech")

## The threshold

Raising it keeps only the frames the model is sure of; the median filter
then decides how short a pause or a blip may be.

In [ ]:
from espnet2.bin.diar_inference import frames_to_segments

for threshold in (0.3, 0.5, 0.7):
    for median in (1, 11):
        found = frames_to_segments(probs, offline.frame_shift, threshold, median)
        speech = sum(s.duration for s in found)
        print(f"threshold {threshold}, median {median:2d}: "
              f"{len(found)} segments, {speech:.2f} s of speech")

## Where next

- **From a terminal**: `espnet detect recording.wav` (offline), `espnet detect *.wav`
  for a batch, or `espnet detect --live` (online) for the microphone
- **From an agent**: the `detect` tool of `espnet-mcp`
- **In the browser**: the Space in
  [`egs2/frontia_vad_v1/diar1/demo`](https://github.com/espnet/espnet/tree/master/egs2/frontia_vad_v1/diar1/demo),
  with online and offline drawn on the waveform
- **Training your own**: the recipe,
  [`egs2/frontia_vad_v1/diar1`](https://github.com/espnet/espnet/tree/master/egs2/frontia_vad_v1/diar1)